In [ ]:
!pip install transformers datasets torch

In [ ]:
from transformers import AutoTokenizer

model_adi = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_adi)

ornek_cumle = "ürün henüz yeni geldi sıcak üflemesi çalışmıyor zaten iyi paketleme durumu yok gönderirken azından kontrol edilebilir"

tokenlar = tokenizer.tokenize(ornek_cumle)
print("Token'lara Ayrılmış Cümle:")
print(tokenlar)

girdiler = tokenizer(ornek_cumle, padding=True, truncation=True, return_tensors="pt")

print("\nBERT'in Göreceği Sayısal Kimlikler (Input IDs):")
print(girdiler["input_ids"])

print("\nDikkat Maskesi (Attention Mask):")
print(girdiler["attention_mask"])

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

Token'lara Ayrılmış Cümle:
['ürün', 'henüz', 'yeni', 'geldi', 'sıcak', 'üf', '##lemesi', 'çalışmıyor', 'zaten', 'iyi', 'paketleme', 'durumu', 'yok', 'gönderir', '##ken', 'azından', 'kontrol', 'edilebilir']

BERT'in Göreceği Sayısal Kimlikler (Input IDs):
tensor([[    2,  2782,  5079,  2360,  3381,  3981, 22020,  7909, 22576,  4154,
          2395, 24849,  4572,  2407, 22116,  2205,  6847,  3248,  7859,     3]])

Dikkat Maskesi (Attention Mask):
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

df = pd.read_csv("temizlenmis_yorumlar.csv")
df['Final_Review'] = df['Final_Review'].fillna("")

df['Sentiment'] = pd.to_numeric(df['Sentiment'], errors='coerce').fillna(0).astype(int)

class YorumDataset(Dataset):
    def __init__(self, metinler, etiketler, tokenizer, max_uzunluk):
        self.metinler = metinler
        self.etiketler = etiketler
        self.tokenizer = tokenizer
        self.max_uzunluk = max_uzunluk

    def __len__(self):
        return len(self.metinler)

    def __getitem__(self, index):
        metin = str(self.metinler[index])
        etiket = self.etiketler[index]

        encoding = self.tokenizer(
            metin,
            add_special_tokens=True,
            max_length=self.max_uzunluk,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(etiket, dtype=torch.long)
        }

X = df['Final_Review'].to_numpy()
y = df['Sentiment'].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

MAX_LEN = 128
BATCH_SIZE = 16

train_dataset = YorumDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset = YorumDataset(X_val, y_val, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

ilk_batch = next(iter(train_loader))
print("Input IDs Boyutu:", ilk_batch['input_ids'].shape)
print("Attention Mask Boyutu:", ilk_batch['attention_mask'].shape)
print("Hedef Etiketler Boyutu:", ilk_batch['targets'].shape)

Input IDs Boyutu: torch.Size([16, 128])
Attention Mask Boyutu: torch.Size([16, 128])
Hedef Etiketler Boyutu: torch.Size([16])


In [ ]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan donanım: {device}")

model = BertForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    num_labels=2
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 2

for epoch in range(epochs):
    model.train()
    toplam_kayip = 0

    loop = tqdm(train_loader, leave=True, position=0)

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['targets'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        toplam_kayip += loss.item()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    ortalama_kayip = toplam_kayip / len(train_loader)
    print(f"\nEpoch {epoch+1} Tamamlandı. Ortalama Kayıp: {ortalama_kayip:.4f}")

Kullanılan donanım: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1: 1


Epoch 1 Tamamlandı. Ortalama Kayıp: 0.0947


Epoch 2: 100%|██████████| 1884/1884 [11:34<00:00,  2.71it/s, loss=0.00201]


Epoch 2 Tamamlandı. Ortalama Kayıp: 0.0555


In [ ]:
model.save_pretrained("./kaydedilmis_bert_modeli")
tokenizer.save_pretrained("./kaydedilmis_bert_modeli")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./kaydedilmis_bert_modeli/tokenizer_config.json',
 './kaydedilmis_bert_modeli/tokenizer.json')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

model.eval()
gercek_etiketler = []
tahminler = []

with torch.no_grad():
  for batch in val_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["targets"].to(device)

    outputs = model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    batch_tahminler = torch.argmax(logits, dim=1).cpu().numpy()
    batch_gercekler = labels.cpu().numpy()

    tahminler.extend(batch_tahminler)
    gercek_etiketler.extend(batch_gercekler)

print(classification_report(gercek_etiketler, tahminler, target_names=["Olumsuz (0)", "Olumlu (1)"]))

              precision    recall  f1-score   support

 Olumsuz (0)       0.90      0.52      0.66       143
  Olumlu (1)       0.98      1.00      0.99      3205

    accuracy                           0.98      3348
   macro avg       0.94      0.76      0.82      3348
weighted avg       0.98      0.98      0.97      3348



In [ ]:
import shap
import torch
import scipy as sp
import numpy as np

def f(x):
    tv = torch.tensor([tokenizer.encode(v, padding='max_length', max_length=128, truncation=True) for v in x]).cuda()
    outputs = model(tv)[0].detach().cpu().numpy()
    scores = (np.exp(outputs).T / np.exp(outputs).sum(-1)).T
    return scores

explainer = shap.Explainer(f, tokenizer)

test_cumlesi = ["ürün henüz yeni geldi sıcak üflemesi çalışmıyor zaten iyi paketleme durumu yok gönderirken azından kontrol edilebilir"]

shap_values = explainer(test_cumlesi)

shap.plots.text(shap_values)